# NiOA-Optimised Bidirectional DRNN — Multi-Horizon Energy Prediction

**Author** : Anwesha Singh  
**Dept.**   : Computer Science Engineering, Manipal University Jaipur

---

## Purpose

This notebook implements the complete training pipeline for the proposed model:

1. Data loading and preprocessing  
2. Strictly chronological train / validation / test splitting  
3. Feature scaling (train-only scaler)  
4. Sliding-window sequence generation  
5. **Saving canonical splits** — used identically by all benchmark models  
6. NiOA hyperparameter optimisation on a 30 % training subset  
7. Final model training with the best-found hyperparameters  
8. Evaluation and artefact saving (model, scaler, predictions, metrics, plots)  

Run cells **sequentially**.  
Set `HORIZON` in Cell 2 to the desired forecast horizon (1, 60, 300, 900, or 1800 seconds),  
then restart the kernel and re-run for each horizon.


## Cell 1 — Environment Setup and Imports

In [ ]:
import os, sys, json, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import tensorflow as tf
import tensorflow.keras.backend as K
from datetime import datetime
from tensorflow.keras.callbacks import EarlyStopping

# ---------------------------------------------------------------------------
# Make core package importable from this notebook's location.
# Adjust PROJECT_ROOT if the notebook is moved to a different directory.
# ---------------------------------------------------------------------------
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
CORE_DIR     = os.path.join(PROJECT_ROOT, "core")
for _p in [PROJECT_ROOT, CORE_DIR]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

from core.config        import *
from core.utils         import set_seeds, configure_gpu, scale_numeric_features, create_sequences, to_python_types, make_tf_dataset
from core.preprocessing import load_and_prepare_data, split_by_timestamp
from core.models        import NinjaOptimizationAlgorithm, create_lstm_model
from core.train         import objective_function_lstm
from core.evaluate      import evaluate_keras_model

configure_gpu()
set_seeds(RANDOM_SEED)
sns.set(style="whitegrid")

print(f"TensorFlow : {tf.__version__}")
print(f"Project root : {PROJECT_ROOT}")

## Cell 2 — Select Forecast Horizon

In [ ]:
# ============================================================
#  SET THIS VALUE before running the rest of the notebook.
#  Valid choices: 1, 60, 300, 900, 1800  (seconds)
# ============================================================
HORIZON = 60

assert HORIZON in FORECAST_HORIZONS, (
    f"HORIZON must be one of {FORECAST_HORIZONS}.  Got {HORIZON}."
)
print(f"Forecast horizon : k = {HORIZON} seconds")

## Cell 3 — Data Loading and Preprocessing

In [ ]:
print(f"Loading data from : {DATA_PATH}")
df = load_and_prepare_data(DATA_PATH, k=HORIZON)

print(f"Dataset shape after preprocessing : {df.shape}")
print(f"Date range : {df['server_timestamp'].min()} -> {df['server_timestamp'].max()}")
print()
print("Target statistics (energy_delta_k):")
print(df['energy_delta_k'].describe())

## Cell 4 — Chronological Train / Validation / Test Split

In [ ]:
t_train_end = df["server_timestamp"].quantile(TRAIN_RATIO)
t_val_end   = df["server_timestamp"].quantile(TRAIN_RATIO + VAL_RATIO)

train_df, val_df, test_df = split_by_timestamp(df, t_train_end, t_val_end)

# Sanity checks — these assertions will raise immediately if there is
# any chronological ordering violation.
assert train_df["server_timestamp"].max() < val_df["server_timestamp"].min(),  "Leakage detected: train overlaps validation!"
assert val_df["server_timestamp"].max()   < test_df["server_timestamp"].min(), "Leakage detected: validation overlaps test!"

print("Split verified — strictly chronological. No leakage.")
print(f"  Train      : {len(train_df):>10,} rows  "
      f"({train_df['server_timestamp'].min()} → {train_df['server_timestamp'].max()})")
print(f"  Validation : {len(val_df):>10,} rows  "
      f"({val_df['server_timestamp'].min()} → {val_df['server_timestamp'].max()})")
print(f"  Test       : {len(test_df):>10,} rows  "
      f"({test_df['server_timestamp'].min()} → {test_df['server_timestamp'].max()})")

## Cell 5 — Feature Scaling and Sequence Generation

In [ ]:
TARGET      = "energy_delta_k"
feature_cols = [
    c for c in train_df.columns
    if c not in ["server_timestamp", "energy", TARGET]
]

X_train_raw = train_df[feature_cols].copy()
y_train_raw = train_df[TARGET].values
X_val_raw   = val_df[feature_cols].copy()
y_val_raw   = val_df[TARGET].values
X_test_raw  = test_df[feature_cols].copy()
y_test_raw  = test_df[TARGET].values

# Scale features — scaler fitted ONLY on training data
X_train_sc, X_val_sc, X_test_sc, scaler = scale_numeric_features(
    X_train_raw, X_val_raw, X_test_raw, feature_cols
)

# Sliding-window sequences
X_train_seq, y_train_seq = create_sequences(X_train_sc.values, y_train_raw, SEQUENCE_LENGTH)
X_val_seq,   y_val_seq   = create_sequences(X_val_sc.values,   y_val_raw,   SEQUENCE_LENGTH)
X_test_seq,  y_test_seq  = create_sequences(X_test_sc.values,  y_test_raw,  SEQUENCE_LENGTH)

SEQ_LEN   = X_train_seq.shape[1]
NUM_FEATS = X_train_seq.shape[2]

print(f"Feature columns ({len(feature_cols)}) : {feature_cols}")
print(f"\nSequence shapes")
print(f"  Train      : {X_train_seq.shape}")
print(f"  Validation : {X_val_seq.shape}")
print(f"  Test       : {X_test_seq.shape}")

## Cell 6 — Save Canonical Splits

These arrays are saved once and loaded identically by every benchmark model  
to guarantee a perfectly fair comparison.

In [ ]:
horizon_split_dir = os.path.join(SPLITS_DIR, f"horizon_{HORIZON}")
os.makedirs(horizon_split_dir, exist_ok=True)

np.save(os.path.join(horizon_split_dir, "X_train.npy"),   X_train_seq)
np.save(os.path.join(horizon_split_dir, "y_train.npy"),   y_train_seq)
np.save(os.path.join(horizon_split_dir, "X_val.npy"),     X_val_seq)
np.save(os.path.join(horizon_split_dir, "y_val.npy"),     y_val_seq)
np.save(os.path.join(horizon_split_dir, "X_test.npy"),    X_test_seq)
np.save(os.path.join(horizon_split_dir, "y_test.npy"),    y_test_seq)
joblib.dump(scaler, os.path.join(horizon_split_dir, "scaler.pkl"))

# Save feature column list for reproducibility
with open(os.path.join(horizon_split_dir, "feature_cols.json"), "w") as f:
    json.dump(feature_cols, f, indent=2)

# Save split timestamps for audit trail
split_meta = {
    "horizon_k"          : HORIZON,
    "sequence_length"    : SEQUENCE_LENGTH,
    "train_ratio"        : TRAIN_RATIO,
    "val_ratio"          : VAL_RATIO,
    "train_start"        : str(train_df["server_timestamp"].min()),
    "train_end"          : str(train_df["server_timestamp"].max()),
    "val_start"          : str(val_df["server_timestamp"].min()),
    "val_end"            : str(val_df["server_timestamp"].max()),
    "test_start"         : str(test_df["server_timestamp"].min()),
    "test_end"           : str(test_df["server_timestamp"].max()),
    "n_train_sequences"  : int(len(X_train_seq)),
    "n_val_sequences"    : int(len(X_val_seq)),
    "n_test_sequences"   : int(len(X_test_seq)),
    "n_features"         : int(NUM_FEATS),
    "random_seed"        : RANDOM_SEED,
    "generated_at"       : datetime.now().isoformat(),
}
with open(os.path.join(horizon_split_dir, "split_metadata.json"), "w") as f:
    json.dump(split_meta, f, indent=2)

print(f"Canonical splits saved to : {horizon_split_dir}")
for fname in os.listdir(horizon_split_dir):
    print(f"  {fname}")

## Cell 7 — NiOA Hyperparameter Optimisation

In [ ]:
n_train_opt = int(len(X_train_seq) * OPT_SUBSET_RATIO)
n_val_opt   = int(len(X_val_seq)   * OPT_SUBSET_RATIO)

X_tr_opt = X_train_seq[:n_train_opt]
y_tr_opt = y_train_seq[:n_train_opt]
X_va_opt = X_val_seq[:n_val_opt]
y_va_opt = y_val_seq[:n_val_opt]

print(f"NiOA optimisation subset sizes : train={n_train_opt:,}  val={n_val_opt:,}")
print(f"Agents={N_AGENTS}  Iterations={MAX_ITERATIONS}  Time limit/trial={OPT_TIME_LIMIT}s")

ninja = NinjaOptimizationAlgorithm(
    objective_function = lambda p: objective_function_lstm(
        p, X_tr_opt, y_tr_opt, X_va_opt, y_va_opt,
        opt_epochs   = OPT_EPOCHS,
        opt_patience = OPT_PATIENCE,
        time_limit   = OPT_TIME_LIMIT,
    ),
    bounds              = HYPERPARAMETER_BOUNDS,
    n_agents            = N_AGENTS,
    max_iterations      = MAX_ITERATIONS,
    exploration_factor  = EXPLORATION_FACTOR,
    exploitation_factor = EXPLOITATION_FACTOR,
)

best_params, best_loss, convergence = ninja.optimize()

print(f"\nBest validation MSE : {best_loss:.8f}")
print("Best hyperparameters :")
for k_name, v in best_params.items():
    print(f"  {k_name} : {v}")

## Cell 8 — Create Experiment Directory and Save Optimisation Artefacts

In [ ]:
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
exp_dir = os.path.join(NIOA_RESULTS_DIR, f"k{HORIZON}_seq{SEQUENCE_LENGTH}_{ts}")

for sub in ["model", "predictions", "plots"]:
    os.makedirs(os.path.join(exp_dir, sub), exist_ok=True)

# Best hyperparameters
with open(os.path.join(exp_dir, "best_params.json"), "w") as f:
    json.dump(to_python_types(best_params), f, indent=4)

# NiOA convergence trace
np.save(os.path.join(exp_dir, "convergence.npy"), np.array(convergence))

print(f"Experiment directory created : {exp_dir}")

## Cell 9 — Final Model Training with tf.data Generator

In [ ]:
K.clear_session()
gc.collect()

model = create_lstm_model(best_params, SEQ_LEN, NUM_FEATS)
model.summary()

# Use tf.data for memory-efficient batch delivery
batch_size = best_params["batch_size"]

train_ds = make_tf_dataset(X_train_seq, y_train_seq, SEQ_LEN, NUM_FEATS, batch_size, shuffle=False)
val_ds   = make_tf_dataset(X_val_seq,   y_val_seq,   SEQ_LEN, NUM_FEATS, batch_size, shuffle=False)

callbacks = [
    EarlyStopping(
        monitor              = "val_loss",
        patience             = FINAL_PATIENCE,
        restore_best_weights = True,
        verbose              = 1,
    )
]

steps_per_epoch  = len(X_train_seq) // batch_size
validation_steps = len(X_val_seq)   // batch_size

history = model.fit(
    train_ds,
    validation_data  = val_ds,
    epochs           = FINAL_EPOCHS,
    steps_per_epoch  = steps_per_epoch,
    validation_steps = validation_steps,
    callbacks        = callbacks,
    verbose          = 1,
)

print("Final training complete.")

## Cell 10 — Save Model and Scaler

In [ ]:
model.save(os.path.join(exp_dir, "model", f"NiOA_DRNN_k{HORIZON}.h5"))
joblib.dump(scaler, os.path.join(exp_dir, "scaler.pkl"))

training_config = {
    "horizon_k"        : HORIZON,
    "sequence_length"  : SEQUENCE_LENGTH,
    "train_ratio"      : TRAIN_RATIO,
    "val_ratio"        : VAL_RATIO,
    "random_seed"      : RANDOM_SEED,
    "opt_subset_ratio" : OPT_SUBSET_RATIO,
    "n_agents"         : N_AGENTS,
    "max_iterations"   : MAX_ITERATIONS,
    "final_epochs"     : FINAL_EPOCHS,
    "final_patience"   : FINAL_PATIENCE,
    "best_params"      : to_python_types(best_params),
    "best_nioa_loss"   : float(best_loss),
    "completed_at"     : datetime.now().isoformat(),
}
with open(os.path.join(exp_dir, "training_config.json"), "w") as f:
    json.dump(training_config, f, indent=4)

print("Model, scaler, and training config saved.")

## Cell 11 — Evaluation on Held-Out Test Set

In [ ]:
metrics_df, y_true, y_pred, note = evaluate_keras_model(model, X_test_seq, y_test_seq)

print(metrics_df.to_string(index=False))
print(f"\nNote: {note}")

# Save predictions
np.save(os.path.join(exp_dir, "predictions", "y_test_true.npy"), y_true)
np.save(os.path.join(exp_dir, "predictions", "y_test_pred.npy"), y_pred)

# Save metrics
metrics_dict = dict(zip(metrics_df["Metric"], metrics_df["Value"]))
with open(os.path.join(exp_dir, "metrics.json"), "w") as f:
    json.dump(metrics_dict, f, indent=4)

print("Evaluation artefacts saved.")

## Cell 12 — Publication-Quality Plots

In [ ]:
plots_dir = os.path.join(exp_dir, "plots")

# ── 1. Predicted vs Actual ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_true, y_pred, alpha=0.4, s=5, color="steelblue", label="Samples")
lim = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
ax.plot(lim, lim, "r--", linewidth=1.5, label="Perfect prediction")
ax.set_xlabel(f"Actual  ΔE$_{{k={HORIZON}}}$ (kWh)")
ax.set_ylabel(f"Predicted  ΔE$_{{k={HORIZON}}}$ (kWh)")
ax.set_title(f"Predicted vs Actual — k = {HORIZON} s")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "pred_vs_actual.png"), dpi=200)
plt.show()
plt.close()

# ── 2. Residual Distribution ────────────────────────────────────────────────
residuals = y_true - y_pred
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(residuals, bins=50, kde=True, ax=ax, color="steelblue")
ax.axvline(0, color="red", linestyle="--", linewidth=1.2)
ax.set_xlabel("Residual (kWh)")
ax.set_title(f"Residual Distribution — k = {HORIZON} s")
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "residuals.png"), dpi=200)
plt.show()
plt.close()

# ── 3. Training / Validation Loss Curve ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history.history["loss"],     label="Training loss",    linewidth=1.5)
ax.plot(history.history["val_loss"], label="Validation loss",  linewidth=1.5)
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE")
ax.set_title(f"Training / Validation Loss — k = {HORIZON} s")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "training_curve.png"), dpi=200)
plt.show()
plt.close()

# ── 4. NiOA Convergence Curve ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(convergence, marker="o", linewidth=2, color="darkorange", markersize=6)
ax.set_xlabel("Iteration (0 = initial population)")
ax.set_ylabel("Best Validation MSE")
ax.set_title(f"NiOA Convergence — k = {HORIZON} s")
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "nioa_convergence.png"), dpi=200)
plt.show()
plt.close()

# ── 5. Time-Series Overlay (first 500 test points) ──────────────────────────
n_show = min(500, len(y_true))
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(y_true[:n_show], label="Actual",    linewidth=1,   alpha=0.9)
ax.plot(y_pred[:n_show], label="Predicted", linewidth=1,   alpha=0.7, linestyle="--")
ax.set_xlabel("Test sample index")
ax.set_ylabel(f"ΔE$_{{k={HORIZON}}}$ (kWh)")
ax.set_title(f"Actual vs Predicted — First {n_show} Test Samples  (k = {HORIZON} s)")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "time_series_overlay.png"), dpi=200)
plt.show()
plt.close()

print(f"All plots saved to : {plots_dir}")

## Cell 13 — Summary

In [ ]:
print("=" * 65)
print(f"  NiOA-DRNN  |  Horizon k = {HORIZON} s  |  seq_len = {SEQUENCE_LENGTH}")
print("=" * 65)
print(metrics_df.to_string(index=False))
print("-" * 65)
print(f"Best NiOA validation MSE : {best_loss:.8f}")
print(f"Experiment artefacts     : {exp_dir}")
print(f"Canonical splits         : {horizon_split_dir}")
print("=" * 65)